In [0]:
# ==========================================
# 03. SLOWLY CHANGING DIMENSIONS (SCD 1 & SCD 2)
# ==========================================
from delta.tables import DeltaTable
from pyspark.sql.functions import col, trim, current_timestamp, lit, row_number
from pyspark.sql.window import Window

catalog = "workspace"
schema_name = "default"
bronze_table = f"{catalog}.{schema_name}.wiki_bronze_data"
silver_scd1_table = f"{catalog}.{schema_name}.wiki_silver_scd1"
silver_scd2_table = f"{catalog}.{schema_name}.wiki_silver_scd2"

# Read & clean input data
df_bronze = spark.table(bronze_table)
df_cleaned = df_bronze.filter(col("id").isNotNull()).withColumn("_ingestion_time", current_timestamp())
window_spec = Window.partitionBy("id", "timestamp").orderBy(col("_ingestion_time").desc())
df_deduplicated = df_cleaned.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1).drop("row_num")

# --- 1. SCD TYPE 1 IMPLEMENTATION ---
if not spark.catalog.tableExists(silver_scd1_table):
    df_deduplicated.write.format("delta").mode("overwrite").saveAsTable(silver_scd1_table)
else:
    delta_scd1 = DeltaTable.forName(spark, silver_scd1_table)
    delta_scd1.alias("target").merge(df_deduplicated.alias("source"), "target.id = source.id") \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# --- 2. SCD TYPE 2 IMPLEMENTATION ---
df_scd2_source = df_deduplicated \
    .withColumn("valid_from", current_timestamp()) \
    .withColumn("valid_to", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

if not spark.catalog.tableExists(silver_scd2_table):
    df_scd2_source.write.format("delta").mode("overwrite").saveAsTable(silver_scd2_table)
else:
    delta_scd2 = DeltaTable.forName(spark, silver_scd2_table)
    delta_scd2.alias("target").merge(df_scd2_source.alias("source"), "target.id = source.id AND target.is_current = true") \
        .whenMatchedUpdate(set={"valid_to": "source.valid_from", "is_current": "false"}).execute()
    df_scd2_source.write.format("delta").mode("append").saveAsTable(silver_scd2_table)

print("✅ Step 3 Complete! SCD Type 1 & Type 2 executed via MERGE.")
display(spark.table(silver_scd1_table))